# Ask-SLM CLI Runner (No UI)

This notebook runs Ask-SLM CLI capabilities using a selected workspace only.

Usage flow:
1. Set `WORKSPACE` in Cell 2.
2. Run one or more analysis cells below.
3. Outputs are written into `Workspaces/<WORKSPACE>/results/` by the CLI defaults.

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

# Configure this once per run.
WORKSPACE = "ABX"
WORKSPACE_ROOT = Path("Workspaces")


def run_cli(args: list[str], check: bool = True) -> subprocess.CompletedProcess:
    """Run askslm.py with pretty command logging."""
    cmd = [sys.executable, "askslm.py"] + args
    print("$", " ".join(shlex.quote(part) for part in cmd))
    result = subprocess.run(cmd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


workspace_dir = WORKSPACE_ROOT / WORKSPACE
workspace_inputs = workspace_dir / "inputs"
workspace_configs = workspace_dir / "mapping_configs"
workspace_results = workspace_dir / "results"

print(f"Workspace: {workspace_dir}")
print(f"Inputs:    {workspace_inputs}")
print(f"Configs:   {workspace_configs}")
print(f"Results:   {workspace_results}")

if not workspace_dir.exists():
    raise FileNotFoundError(f"Workspace not found: {workspace_dir}")

In [11]:
# Metadata overview
run_cli([
    "metadata-overview",
    "--view", WORKSPACE,
])

Building metadata overview report ...
  → Workspaces/ABX/results/metadata_overview/metadata_overview_summary.csv
  → Workspaces/ABX/results/metadata_overview/metadata_overview_projects.csv
  → Workspaces/ABX/results/metadata_overview/metadata_overview_discovered_keys_by_project.csv
  → Workspaces/ABX/results/metadata_overview/metadata_overview_discovered_keys_unique.csv

🔍 Discovered Source Keys (57 unique keys across 1 projects)

Metadata Summary:
                              metric value
                             samples   960
                            projects     1
                    metadata_columns    37
                mapped_columns_count     0
                      mapped_columns      
       discovered_source_keys_unique    57
projects_with_discovered_source_keys     1


CompletedProcess(args=['/Users/yarintamam/Code/Ask-SLM/.venv/bin/python', 'askslm.py', 'metadata-overview', '--view', 'ABX'], returncode=0)

In [12]:
# Abundance matrix
run_cli([
    "abundance-matrix",
    "--view", WORKSPACE,
])

$ /Users/yarintamam/Code/Ask-SLM/.venv/bin/python askslm.py abundance-matrix --view ABX
Loading metadata from Workspaces/ABX/inputs/metadata.csv ...
Loading taxonomy from Workspaces/ABX/inputs/taxonomy.csv ...
Loaded: 1284 samples, 2365668 taxon observations
Building abundance matrix (samples × taxa) ...
  → Workspaces/ABX/results/abundance_matrix/abundance_matrix.csv
  Shape: 1284 samples × 23615 taxa


CompletedProcess(args=['/Users/yarintamam/Code/Ask-SLM/.venv/bin/python', 'askslm.py', 'abundance-matrix', '--view', 'ABX'], returncode=0)

In [13]:
# Build taxonomy lineage DB (required by alpha-diversity)
run_cli([
    "build-taxonomy-lineage",
    "--view", WORKSPACE,
])

$ /Users/yarintamam/Code/Ask-SLM/.venv/bin/python askslm.py build-taxonomy-lineage --view ABX
Loading taxonomy from Workspaces/ABX/inputs/taxonomy.csv ...
Loaded: 2365668 taxon observations
✅ Taxonomy lineage cache hit: Workspaces/ABX/results/_cache/taxonomy_lineage_db.json


CompletedProcess(args=['/Users/yarintamam/Code/Ask-SLM/.venv/bin/python', 'askslm.py', 'build-taxonomy-lineage', '--view', 'ABX'], returncode=0)

In [14]:
# Alpha diversity
ALPHA_RANK = "genus"
ALPHA_GROUP = None  # Example: "Group"
LINEAGE_DB_PATH = workspace_results / "_cache" / "taxonomy_lineage_db.json"

cmd = [
    "alpha-diversity",
    "--view", WORKSPACE,
    "--rank", ALPHA_RANK,
    "--lineage-db-path", str(LINEAGE_DB_PATH),
]
if ALPHA_GROUP:
    cmd += ["--group", ALPHA_GROUP]

run_cli(cmd)

$ /Users/yarintamam/Code/Ask-SLM/.venv/bin/python askslm.py alpha-diversity --view ABX --rank genus --lineage-db-path Workspaces/ABX/results/_cache/taxonomy_lineage_db.json
Loading metadata from Workspaces/ABX/inputs/metadata.csv ...
Loading taxonomy from Workspaces/ABX/inputs/taxonomy.csv ...
Loaded: 1284 samples, 2365668 taxon observations
Computing alpha diversity at rank 'genus' ...
✅ Loaded lineage database with 23283 taxa
  → Workspaces/ABX/results/alpha_diversity/alpha_diversity.csv
  → Workspaces/ABX/results/alpha_diversity/alpha_diversity.html

Summary:
  Richness — mean: 409.6, median: 434.0
  Shannon  — mean: 2.30, median: 2.66


CompletedProcess(args=['/Users/yarintamam/Code/Ask-SLM/.venv/bin/python', 'askslm.py', 'alpha-diversity', '--view', 'ABX', '--rank', 'genus', '--lineage-db-path', 'Workspaces/ABX/results/_cache/taxonomy_lineage_db.json'], returncode=0)

In [15]:
# Taxonomic abundance
TAXONOMIC_RANK = "genus"
TAXONOMIC_TAXON = "Prevotella"
TAXONOMIC_GROUP = None  # Example: "Group"

cmd = [
    "taxonomic-abundance",
    "--view", WORKSPACE,
    "--rank", TAXONOMIC_RANK,
    "--taxon", TAXONOMIC_TAXON,
]
if TAXONOMIC_GROUP:
    cmd += ["--group", TAXONOMIC_GROUP]

run_cli(cmd)

$ /Users/yarintamam/Code/Ask-SLM/.venv/bin/python askslm.py taxonomic-abundance --view ABX --rank genus --taxon Prevotella
Loading metadata from Workspaces/ABX/inputs/metadata.csv ...
Loading taxonomy from Workspaces/ABX/inputs/taxonomy.csv ...
Loaded: 1284 samples, 2365668 taxon observations
Computing abundance of 'Prevotella' at rank 'genus' ...
  → Workspaces/ABX/results/taxonomic_abundance/taxonomic_abundance_Prevotella.csv
  → Workspaces/ABX/results/taxonomic_abundance/taxonomic_abundance_Prevotella_summary.csv
  → Workspaces/ABX/results/taxonomic_abundance/taxonomic_abundance_Prevotella.html

Prevalence Summary:
      Cohort  Total Samples (N)  Samples with 0 Reads Samples with Non-Zero Reads (%) Mean Rel. Abundance
All Combined               1284                   244                          81.00%           3.342851%


CompletedProcess(args=['/Users/yarintamam/Code/Ask-SLM/.venv/bin/python', 'askslm.py', 'taxonomic-abundance', '--view', 'ABX', '--rank', 'genus', '--taxon', 'Prevotella'], returncode=0)

In [16]:
# Single taxon abundance
SINGLE_TAXON_RANK = "superkingdom"
SINGLE_TAXON_NAME = "Viruses"
SINGLE_SOURCE_KEYS = ["day_run", "host_subject_id_sam"]

run_cli([
    "single-taxon-abundance",
    "--view", WORKSPACE,
    "--rank", SINGLE_TAXON_RANK,
    "--taxon", SINGLE_TAXON_NAME,
    "--source-keys", *SINGLE_SOURCE_KEYS,
])

$ /Users/yarintamam/Code/Ask-SLM/.venv/bin/python askslm.py single-taxon-abundance --view ABX --rank superkingdom --taxon Viruses --source-keys day_run host_subject_id_sam
Loading metadata from Workspaces/ABX/inputs/metadata.csv ...
Loading taxonomy from Workspaces/ABX/inputs/taxonomy.csv ...
Loaded: 1284 samples, 2365668 taxon observations
Computing single taxon abundance for taxon='Viruses', rank='superkingdom', source_keys=['day_run', 'host_subject_id_sam'] ...
  → Workspaces/ABX/results/single_taxon_abundance/single_taxon_abundance_Viruses_raw.csv
  → Workspaces/ABX/results/single_taxon_abundance/single_taxon_abundance_Viruses_summary.csv
  → Workspaces/ABX/results/single_taxon_abundance/single_taxon_abundance_Viruses.html

Prevalence Summary:
      Cohort  Total Samples (N)  Samples with 0 Reads Samples with Non-Zero Reads (%) Mean Rel. Abundance
All Combined               1284                    59                          95.40%           1.659584%


CompletedProcess(args=['/Users/yarintamam/Code/Ask-SLM/.venv/bin/python', 'askslm.py', 'single-taxon-abundance', '--view', 'ABX', '--rank', 'superkingdom', '--taxon', 'Viruses', '--source-keys', 'day_run', 'host_subject_id_sam'], returncode=0)